In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

import numpy as np
import matplotlib.pyplot as plt

from train import train
from predict import predict
from utils.combine import combine
from utils.plot_loss import plot_loss

In [ ]:
is_test = False

# Load the data
out = combine(is_test)
print(out.shape)

In [ ]:
# Plot the data
plt.figure()
plt.plot(torch.transpose(out[0], 0, 1).numpy())
plt.show()

In [ ]:
# Save the data
np.save(f'./data/{"testing" if is_test else "training"}/{"test" if is_test else "train"}.npy', out.numpy())

In [ ]:
is_test = False

out = torch.from_numpy(np.load(f'./data/{"testing" if is_test else "training"}/{"test" if is_test else "train"}.npy'))
labels = torch.from_numpy(np.load(f'./data/{"testing" if is_test else "training"}/{"test" if is_test else "train"}Labels.npy'))

in_channels = out.shape[1]
out_channels = labels.unique().numel()

In [51]:
class ConvolutionalNeuralNetwork(nn.Module):
    def __init__(self):
        super(ConvolutionalNeuralNetwork, self).__init__()
        
        # Define the convolutional layers
        self.conv1 = nn.Conv1d(
            in_channels, 
            out_channels=64, 
            kernel_size=3,
        )
        self.conv2 = nn.Conv1d(
            in_channels=64, 
            out_channels=92, 
            kernel_size=3,
        )
        
        # Define the pooling layers
        self.fc1 = nn.Linear(73232, 2048)
        self.fc2 = nn.Linear(2048, out_channels)
        
        # Define the activation functions and dropout
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(p=0.5)

    def forward(self, x):
        # Convolutional layers
        x = self.relu(self.conv1(x))
        x = self.relu(self.conv2(x))
        
        # Flatten the output of the convolutional layers
        x = x.view(x.size(0), -1)
        
        # Fully connected layers
        x = self.relu(self.fc1(x))
        x = self.dropout(x)
        x = self.fc2(x)
        
        return x

ConvolutionalNeuralNetwork(
  (conv1): Conv1d(27, 64, kernel_size=(3,), stride=(1,))
  (conv2): Conv1d(64, 92, kernel_size=(3,), stride=(1,))
  (fc1): Linear(in_features=73232, out_features=2048, bias=True)
  (fc2): Linear(in_features=2048, out_features=55, bias=True)
  (relu): ReLU()
  (dropout): Dropout(p=0.5, inplace=False)
)


In [ ]:
# Train the model
model = ConvolutionalNeuralNetwork()

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model.parameters(), lr=1e-4, weight_decay=1e-5)

number_of_epochs = 2
batch_size = 32

losses = train(model, optimizer, criterion, out, labels, batch_size, number_of_epochs)
plot_loss(losses)

In [ ]:
# Load the test data
is_test = True
out = combine(is_test)
print(out.shape)

# out = torch.from_numpy(np.load(f'./data/{"testing" if is_test else "training"}/{"test" if is_test else "train"}.npy'))
labels = torch.from_numpy(np.load(f'./data/{"testing" if is_test else "training"}/{"test" if is_test else "train"}Labels.npy'))

In [ ]:
# Load the model
# model = ConvolutionalNeuralNetwork()
# model.load_state_dict(torch.load('model.pth', weights_only=True))

In [ ]:
predictions = predict(model, out, labels, batch_size)
print(predictions.shape)

In [ ]:
# Evaluate the model
# Convert predictions to class labels
predicted = torch.argmax(predictions, dim=1)
labels = labels[:predicted.size(0)]

# Compute accuracy
from eval.accuracy import accuracy
print(f'Accuracy of the model on the test data: {accuracy(predicted, labels):.2f}%')

# Compute average F1 score
from eval.average_f1_score import average_f1_score
print(f'Average F1 score: {average_f1_score(predicted, labels):.2f}')

# Compute mean average precision
from eval.mean_average_precision import computeMeanAveragePrecision as mean_average_precision
print(f'Mean Average Precision: {mean_average_precision(labels, predictions)[0]:.2f}')

# Compute heat map
from sklearn.metrics import confusion_matrix
def compute_confusion_matrix(predictions, labels):
    cm = confusion_matrix(labels.numpy(), predictions.numpy())

    plt.imshow(cm, cmap='Blues', interpolation='nearest')
    plt.show()
    
compute_confusion_matrix(predicted, labels)

In [ ]:
# Store the model
import datetime
torch.save(model.state_dict(), f'./models/model-{datetime.datetime.now().strftime("%Y-%m-%d-%H-%M")}--{accuracy(predicted, labels):.2f}-{average_f1_score(predicted, labels):.2f}.pth')

In [ ]:
accuracy_with_random_chance = 100 / labels.unique().numel()
print(f'Accuracy with random chance: {accuracy_with_random_chance:.2f}%')